In [1]:
# подключаем нужные библиотеки
import os
import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# скачиваем данные о визитах и регистрациях
API_URL = os.getenv('API_URL')
DATE_BEGIN = os.getenv('DATE_BEGIN')
DATE_END = os.getenv('DATE_END')

url_visits = API_URL + "/visits?begin=" + DATE_BEGIN + "&end=" + DATE_END
url_regs = API_URL + "/registrations?begin=" + DATE_BEGIN + "&end=" + DATE_END

df_api_visits = pd.DataFrame(requests.get(url_visits).json())
df_api_regs = pd.DataFrame(requests.get(url_regs).json())


In [3]:
# убираем ботов и дубликаты, считаем общую конверсию по дням
df_visits_clean = df_api_visits[~df_api_visits['user_agent'].str.contains('bot', case=False, na=False)].copy()

df_visits_clean = df_visits_clean.sort_values('datetime')
df_visits_clean = df_visits_clean.drop_duplicates(subset='visit_id', keep='last')

df_visits_clean['date_group'] = pd.to_datetime(df_visits_clean['datetime']).dt.date
df_api_regs['date_group'] = pd.to_datetime(df_api_regs['datetime']).dt.date

visits_grouped = df_visits_clean.groupby(['date_group', 'platform']).size().reset_index(name='visits')
regs_grouped = df_api_regs.groupby(['date_group', 'platform']).size().reset_index(name='registrations')

df_conversion = pd.merge(visits_grouped, regs_grouped, on=['date_group', 'platform'], how='outer')

df_conversion['visits'] = df_conversion['visits'].fillna(0).astype(int)
df_conversion['registrations'] = df_conversion['registrations'].fillna(0).astype(int)

df_conversion['conversion'] = (df_conversion['registrations'] / df_conversion['visits'] * 100).fillna(0)
df_conversion = df_conversion.sort_values('date_group').reset_index(drop=True)

df_conversion.to_json('./conversion.json')

df_conversion.head(10)

,date_group,platform,visits,registrations,conversion
0,2023-03-01,android,75,61,81.333333
1,2023-03-01,ios,22,18,81.818182
2,2023-03-01,web,279,8,2.867384
3,2023-03-02,android,67,59,88.059701
4,2023-03-02,ios,31,24,77.419355
5,2023-03-02,web,515,23,4.466019
6,2023-03-03,android,26,22,84.615385
7,2023-03-03,ios,40,34,85.000000
8,2023-03-03,web,617,51,8.265802
9,2023-03-04,android,94,77,81.914894


In [4]:
# загружаем файл с расходами на рекламу и соединяем таблицы
df_api_ads = pd.read_csv('ads.csv')

df_api_ads['date_group'] = pd.to_datetime(df_api_ads['date']).dt.date

ads_grouped = df_api_ads.groupby(['date_group', 'utm_campaign'])['cost'].sum().reset_index()

df_conversion_clean = df_conversion.groupby('date_group')[['visits', 'registrations']].sum().reset_index()

df_ads_final = pd.merge(df_conversion_clean, ads_grouped, on='date_group', how='outer')

df_ads_final['visits'] = df_ads_final['visits'].fillna(0).astype(int)
df_ads_final['registrations'] = df_ads_final['registrations'].fillna(0).astype(int)
df_ads_final['cost'] = df_ads_final['cost'].fillna(0).astype(int)
df_ads_final['utm_campaign'] = df_ads_final['utm_campaign'].fillna('none')

df_ads_final = df_ads_final[['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']]
df_ads_final = df_ads_final.sort_values('date_group').reset_index(drop=True)

df_ads_final.to_json('./ads.json')

df_ads_final.head(10)

,date_group,visits,registrations,cost,utm_campaign
0,2023-03-01,376,87,212,advanced_algorithms_series
1,2023-03-02,613,106,252,advanced_algorithms_series
2,2023-03-03,683,107,202,advanced_algorithms_series
3,2023-03-04,647,159,223,advanced_algorithms_series
4,2023-03-05,707,115,265,advanced_algorithms_series
5,2023-03-06,1291,230,108,advanced_algorithms_series
6,2023-03-07,1382,124,165,advanced_algorithms_series
7,2023-03-08,1382,151,155,advanced_algorithms_series
8,2023-03-09,1064,209,124,advanced_algorithms_series
9,2023-03-10,812,112,276,advanced_algorithms_series


In [5]:
# строим графики 
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

df_daily_total = df_ads_final.groupby('date_group')[['visits', 'registrations', 'cost']].sum().reset_index()

# 1. Итоговые визиты
plt.figure()
sns.lineplot(data=df_daily_total, x='date_group', y='visits', marker='o', color='blue')
plt.title('Итоговые визиты по дням')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/total_visits.png')
plt.close()

# 2. Итоговые визиты с разбивкой по платформам
plt.figure()
sns.lineplot(data=df_conversion, x='date_group', y='visits', hue='platform', marker='o')
plt.title('Итоговые визиты с разбивкой по платформам')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/visits_by_platform.png')
plt.close()

# 3. Итоговые регистрации
plt.figure()
sns.lineplot(data=df_daily_total, x='date_group', y='registrations', marker='o', color='green')
plt.title('Итоговые регистрации по дням')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/total_registrations.png')
plt.close()

# 4. Итоговые регистрации с разбивкой по платформам
plt.figure()
sns.lineplot(data=df_conversion, x='date_group', y='registrations', hue='platform', marker='o')
plt.title('Итоговые регистрации с разбивкой по платформам')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/registrations_by_platform.png')
plt.close()

# 5. Конверсия по каждой платформе
plt.figure()
sns.lineplot(data=df_conversion, x='date_group', y='conversion', hue='platform', marker='o')
plt.title('Конверсия по каждой платформе (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/conversion_by_platform.png')
plt.close()

# 6. Средняя конверсия
df_daily_total['total_conversion'] = (df_daily_total['registrations'] / df_daily_total['visits'] * 100).fillna(0)
plt.figure()
sns.lineplot(data=df_daily_total, x='date_group', y='total_conversion', marker='o', color='purple')
plt.title('Средняя общая конверсия по дням (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/average_conversion.png')
plt.close()

# 7. Стоимости реклам
plt.figure()
sns.lineplot(data=df_daily_total, x='date_group', y='cost', marker='o', color='red')
plt.title('Затраты на рекламу по дням')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('./charts/advertising_costs.png')
plt.close()

# 8. Визиты за весь период с цветовым выделением рекламной кампании
plt.figure()
ax8 = sns.barplot(data=df_ads_final, x='date_group', y='visits', hue='utm_campaign', dodge=False)
plt.title('Визиты по дням с выделением рекламных кампаний')
ticks = range(0, len(df_ads_final['date_group']), 14)
plt.xticks(ticks, df_ads_final['date_group'].iloc[ticks], rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('./charts/visits_by_campaign.png')
plt.close()

# 9. Регистрации за весь период с цветовым выделением рекламной кампании
plt.figure() 
ax9 = sns.barplot(data=df_ads_final, x='date_group', y='registrations', hue='utm_campaign', dodge=False) 
plt.title('Регистрации по дням с выделением рекламных кампаний')
plt.xticks(ticks, df_ads_final['date_group'].iloc[ticks], rotation=45) 
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') 
plt.tight_layout() 
plt.savefig('./charts/registrations_by_campaign.png') 
plt.close()